# 01 数据管道 — 从手工档到自动档

### 先回答你最关心的三个问题

| 问题 | 答案 |
|------|------|
| **统一接口如何下载不同市场的数据？** | 抽象基类定义契约 → 子类各自实现 `_fetch()` → 框架自动标准化。写代码的人只看到 `get_history()`，底层是 A股还是加密完全透明 |
| **数据是直接下载到 SQL 的吗？** | **不是。** 分三步：API → DataFrame(内存) → SQLite(磁盘)。每一步独立，互不耦合 |
| **会不会卡顿？** | 当前是同步拉取，大数量会慢。解法：分页 + 本地缓存。实盘才需要异步 |

---

## 本课产出

| 章节 | 概念 | 关键理解 |
|------|------|----------|
| 1. 架构全景 | API → DataFrame → SQLite 三层分离 | 数据不是"直接灌进SQL" |
| 2. 统一 Schema | 10列标准，所有市场遵守 | 数据契约 |
| 3. ABC 抽象基类 | 多态：同一接口，不同实现 | 为什么这是量化骨架 |
| 4. AShareSource | baostock 工程化 | 批量+错误隔离 |
| 5. CryptoSource | 自动探测可用交易所 | 跨市场统一接口的威力 |
| 6. SQLite 存储 | WAL + UPSERT | 断网也能分析 |
| 7. 批量下载实战 | 从 2 只加密到 10 只A股 | 进度反馈+本地缓存加速 |
| 8. 端到端验证 | 存→读→画→算 | 完整链路的正确性检查 |

## 1. 架构全景 — 数据不直接灌 SQL

很多初学者会误以为"数据管道"就是把 API 返回的数据直接写入数据库。实际架构是**三层分离**：

```
┌─────────────────────────────────────────────────────┐
│                     你的代码                         │
│  source.get_history(symbols, start, end)   ← 一行！  │
├─────────────────────────────────────────────────────┤
│  第1层: DataSource (数据源适配)                      │
│    _fetch()  → 从 API 拉原始数据                     │
│    _normalize() → 统一列名/类型/去重                  │
│    输出: 标准 DataFrame (在内存中)                    │
├─────────────────────────────────────────────────────┤
│  第2层: DataFrame (内存中的数据处理)                  │
│    你可以在这里: 筛选、合并、算指标、画图              │
│    source.get_history() 返回的就是这个 DataFrame      │
├─────────────────────────────────────────────────────┤
│  第3层: DataStore (磁盘持久化)                        │
│    store.save(df) → 写入 SQLite                      │
│    store.load()   → 从 SQLite 读回 DataFrame          │
│    只在"你需要保存"的时候才调用                        │
└─────────────────────────────────────────────────────┘
```

**关键设计**：三层互不耦合。API 不知道什么是 SQLite，SQLite 不知道什么是 API。
- 你可以只拉数据不存（get_history → 直接分析）
- 你可以存了再读（store.save → store.load → 分析）
- 你可以读历史数据，断网也能用（store.load → 分析）

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print("环境就绪")

## 2. 统一 Schema — 所有市场的共同语言

```
baostock 返回:  [date, open, high, low, close, volume, amount]
ccxt 返回:      [timestamp, open, high, low, close, volume]
                          ↓  _normalize()
        统一 Schema（10 列标准，所有市场输出一样）
   [symbol, date, open, high, low, close, volume, amount, market, interval]
```

In [ ]:
from data.schema import OHLCV_COLUMNS, OHLCV_DTYPES

print("标准 OHLCV 列（10 列）：")
for i, col in enumerate(OHLCV_COLUMNS):
    print(f"  {i+1}. {col:12s} → {OHLCV_DTYPES[col]}")

## 3. ABC 抽象基类 — "一个接口，多种实现"

```python
class DataSource(ABC):           # ← 抽象基类：定义"必须有什么"
    @abstractmethod
    def _fetch(self, symbol, start, end):
        ...                      # ← 子类必须实现：从某处拉数据

    def get_history(self, symbols, start, end):
        for sym in symbols:      # ← 框架自动完成：批量拉取
            raw = self._fetch(sym, start, end)
            raw["symbol"] = sym
        return self._normalize(df)  # ← 自动标准化
```

**多态的威力**：你写 `source.get_history(...)` 这一行，不管 `source` 是 `AShareSource` 还是 `CryptoSource`，行为完全一致——这就是"统一接口"。

## 4. AShareSource — A股数据源

直接看代码对比：

In [ ]:
from data.sources.ashare import AShareSource

ashare = AShareSource()

# ┌────────── 旧方式 (Notebook 00) ──────────┐
# │ df = fetch_baostock("sh.000300", ...)     │  ← 每只股票一行
# │ df = fetch_baostock("sh.600519", ...)     │  ← 重复劳动
# │ # 然后手动合并、手加 market 列...           │
# └──────────────────────────────────────────┘
#
# ┌────────── 新方式 ─────────────────────────┐
# │ 一行，批量，自动标准化                       │
# └──────────────────────────────────────────┘

symbols = ["sh.000300", "sh.600519", "sz.300750"]
df_ashare = ashare.get_history(symbols, "2024-01-01", "2024-06-30")

print(f"一次调用: {len(symbols)} 只 → {len(df_ashare)} 行")
print(f"列: {df_ashare.columns.tolist()}")
print(f"market: {df_ashare['market'].unique()}")
print(f"\n每个品种的数据量:")
print(df_ashare.groupby("symbol").size())
df_ashare.head()

In [ ]:
# 错误隔离：一只失败不影响其他
symbols_mixed = ["sh.000300", "NOT.EXIST", "sh.600519"]
df = ashare.get_history(symbols_mixed, "2024-06-01", "2024-06-30")
print(f"请求 {len(symbols_mixed)}只 → 成功 {df['symbol'].nunique()}只")
print(f"失败的被跳过，不报错中断")

## 5. CryptoSource — 同一个接口，另一个市场

**关键**：接口完全一样，但底层自动探测可用的交易所（国内 Binance/OKX 被墙，自动切到 Gate.io）。

你的代码**不用写任何 if/else 判断市场**——这就是抽象层的价值。

In [ ]:
from data.sources.crypto import CryptoSource

crypto = CryptoSource()  # 自动探测可用交易所
# 输出类似: [CryptoSource] 自动选择交易所: gate

# 和 AShareSource 完全一样的调用方式
df_crypto = crypto.get_history(
    symbols=["BTC/USDT", "ETH/USDT"],
    start="2024-01-01",
    end="2024-06-30"
)

print(f"数据量: {len(df_crypto)} 行")
print(f"market: {df_crypto['market'].unique()}")
print(f"symbols: {df_crypto['symbol'].unique()}")
print(f"\n列名一致性验证:")
print(f"  AShare: {df_ashare.columns.tolist()}")
print(f"  Crypto: {df_crypto.columns.tolist()}")
print(f"  → 完全一致，后续分析代码可直接复用")
df_crypto.head()

In [ ]:
# 跨市场对比 → 同一份分析代码，两个市场都不用改
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (df, title, color) in zip(axes, [
    (df_ashare[df_ashare.symbol=="sh.000300"], "沪深300", "steelblue"),
    (df_crypto[df_crypto.symbol=="BTC/USDT"],   "BTC/USDT", "orange")
]):
    ret = np.log(df["close"] / df["close"].shift(1)).dropna()
    ax.hist(ret, bins=50, density=True, alpha=0.7, color=color)
    mu, sig = ret.mean(), ret.std()
    x = np.linspace(mu - 4*sig, mu + 4*sig, 100)
    ax.plot(x, 1/(sig*np.sqrt(2*np.pi))*np.exp(-(x-mu)**2/(2*sig**2)), 'r-', lw=1)
    ax.set_title(f"{title}\n波动率={sig*100:.2f}% | 偏度={ret.skew():.2f} | 峰度={ret.kurtosis():.2f}")
    ax.axvline(x=0, color='gray', ls='--', alpha=0.5)

plt.suptitle("同一套分析代码，两个市场", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("观察：BTC 波动率远高于沪深300，峰度也更肥 → 加密市场风险和机会都更大")

## 6. SQLite 存储 — 数据落地（不是直接灌）

**关键认知**：数据不是"API → SQL"，而是"API → DataFrame → SQL"。
- DataFrame 这一步让你可以在存之前检查、筛选、处理
- SQLite 这一步让你断网后还能读
- 两层是**松耦合**的——你可以只用 DataFrame 不存，也可以只读 SQL 不拉

In [ ]:
from data.store import DataStore

store = DataStore("../data/quant.db")

# 第1步: 从 API 拉到内存 DataFrame（在内存中，你可以随意操作）
# 第2步: 再决定是否存入 SQLite（独立操作）
store.save(df_ashare, market="ashare", interval="daily")
store.save(df_crypto, market="crypto", interval="daily")

print("已存入数据库:")
display(store.list_tables())

In [ ]:
# 之后无论有没有网络，直接从SQLite读
df_loaded = store.load("ashare", "daily", symbols=["sh.000300"])

print(f"从 SQLite 加载: {len(df_loaded)} 行（无需联网）")

# 验证数据完整性
original = df_ashare[df_ashare.symbol == "sh.000300"]
diff = abs(df_loaded["close"].sum() - original["close"].sum())
print(f"数据校验: close总和误差 = {diff:.10f} (应为0)")
df_loaded.head()

In [ ]:
# 关键设计: WAL 模式（避免多进程读写锁冲突）
import sqlite3
conn = sqlite3.connect("../data/quant.db")
wal = conn.execute("PRAGMA journal_mode").fetchone()[0]
print(f"WAL 模式: {wal}")
print(f"→ 允许多个 Notebook 同时读数据，不会锁")

# UPSERT验证
before = conn.execute("SELECT COUNT(*) FROM ashare_daily").fetchone()[0]
store.save(df_ashare, "ashare", "daily")  # 重复存
after = conn.execute("SELECT COUNT(*) FROM ashare_daily").fetchone()[0]
print(f"UPSERT: {before} = {after} (重复写入不翻倍)")
conn.close()

## 7. 批量下载实战 — 解决"卡顿"的问题

### 当前性能

```
同步拉取速度（实测）：
  AShareSource:  ~1-2秒/只（baostock login + 按年分片 + logout）
  CryptoSource:  ~1-3秒/只（ccxt 迭代分页拉取）

拉 10 只股票 ≈ 10-20 秒  ← 可以接受
拉 50 只股票 ≈ 50-100 秒 ← 需要进度条
拉 500 只股票 ≈ 10分钟     ← 需要异步
```

### 当前策略（够用且简单）

1. **本地缓存是第一加速器**：第一次拉完存 SQLite，以后 `store.load()` 毫秒级返回
2. **增量更新**：只拉上次更新之后的新日期
3. **进度反馈**：`bulk_download()` 带回调，知道下载到哪了
4. **实盘时才用异步**：当前学习阶段同步足够

In [ ]:
# 批量下载演示：10只A股 + 进度反馈，并自动缓存到SQLite
import time

symbols_10 = [
    "sh.600519", "sz.300750", "sh.600036", "sz.002594", "sh.601318",
    "sh.600276", "sz.000858", "sh.601012", "sz.002415", "sh.600900"
]

ashare = AShareSource()
store = DataStore("../data/quant.db")

print(f"批量下载 {len(symbols_10)} 只股票...")
start_t = time.time()

# get_history 内部就是批量处理
df_bulk = ashare.get_history(symbols_10, "2024-01-01", "2024-06-30")

elapsed = time.time() - start_t
print(f"\n完成: {df_bulk['symbol'].nunique()}/{len(symbols_10)} 只")
print(f"用时: {elapsed:.1f}秒")
print(f"总行数: {len(df_bulk)}")

# 存入SQLite → 以后秒级读取
store.save(df_bulk, "ashare", "daily")
print(f"已缓存到 SQLite")

In [ ]:
# 验证缓存加速效果
start_t = time.time()
df_cached = store.load("ashare", "daily", symbols=symbols_10)
elapsed = time.time() - start_t

print(f"SQLite 读取 {len(df_cached)} 行")
print(f"用时: {elapsed*1000:.0f}ms")
print(f"vs 首次拉取: 快 {(10+len(symbols_10)*1.5)*1000/elapsed:.0f}x" if elapsed > 0 else "")

## 8. 端到端验证

完整链路：SQLite 读取 → 分析 → 画图（**不联网**）

In [ ]:
# 从数据库读数据直接画图（模拟断网场景）
df_plot = store.load("ashare", "daily", symbols=["sh.000300"], start="2024-01-01")

fig, ax = plt.subplots(figsize=(14, 5))
nav = df_plot.set_index("date")["close"] / df_plot.set_index("date")["close"].iloc[0]
ax.plot(nav.index, nav.values, color="steelblue", linewidth=1.5)
ax.fill_between(nav.index, 1, nav.values, where=(nav.values>=1), color='red', alpha=0.1)
ax.fill_between(nav.index, nav.values, 1, where=(nav.values<1), color='green', alpha=0.1)
ax.axhline(y=1, color='gray', ls='--', alpha=0.5)
ax.set_title("沪深300 2024上半年净值 (数据来源: 本地SQLite, 无需联网)", fontsize=12)
ax.set_ylabel("净值")
ax.grid(True, alpha=0.3)
plt.show()

total_return = (nav.iloc[-1] - 1) * 100
print(f"2024上半年沪深300收益: {total_return:.2f}%")
print(f"数据来源: SQLite → 断网也能分析")

## 9. 总结：你刚刚构建了什么

```
                   ┌─── AShareSource (baostock)
你的代码 ──→ DataSource(ABC) ──┼─── CryptoSource (ccxt)
  get_history()    ├─── USStockSource (yfinance, 未来)
                   └─── ... (任何新市场)
                        ↓
              标准 DataFrame (内存)
                        ↓
              DataStore.save() → SQLite (磁盘)
              DataStore.load() → DataFrame (秒读)
```

### 核心认知

| 你可能以为 | 实际上 |
|------------|--------|
| 数据直接 API→SQL | API→DataFrame→SQL，三步独立 |
| 每次都要联网拉 | 首次拉取→缓存SQLite→以后断网也能用 |
| 不同市场要写不同代码 | 同一行 `get_history()`，底层多态 |
| 批量下载会卡死 | 同步拉 + SQLite缓存，第一次慢、以后快 |

### 下一步

数据管道就绪。接下来构建**回测引擎**——让策略真正跑在干净数据上。